# Walking distance from each metro station to its grocery stores

The 500 m circles and the search distances are straight lines. A straight line crosses buildings, highways and rail yards; a person cannot. This notebook measures the **real walk** along the street network instead:

1. Download Montréal's walking network from OpenStreetMap with **OSMnx** (`network_type="walk"`: sidewalks, footpaths, streets you can walk on).
2. For each station, keep the streets within a **500 m walk** with networkx's `ego_graph(..., distance="length")`, and turn them into an area (the station's **walk area**).
3. Count the open grocery stores inside each walk area with `gpd.sjoin`, and work out each store's walking distance and route.

The page reads the two files this writes: `MetroWalk500m.geojson` (the walk areas) and `MetroWalkRoutes.geojson` (one route per station and store).

A full run takes about 5 to 8 minutes. OSMnx keeps a copy of the download in `notebooks/cache/` (left out of git), so later runs skip the download. If the import says it cannot find osmnx, remove the `#` from the first line, run it, and restart the kernel.

In [1]:
# %pip install osmnx scipy
import os
import geopandas as gpd
import networkx as nx
import osmnx as ox
from shapely.geometry import LineString, Point, Polygon

RADIUS = 500        # metres of walking, the same 500 m the dropdown used as a circle (400 m is about 5 minutes)
STREET_BUFFER = 30  # metres on each side of a reached street, to turn streets into an area
METRIC = "EPSG:2950"   # MTM zone 8, metres

## 1. The stations and the open grocery stores, in metres

The same open stores the page searches (`GroceryNearestMetro.geojson`), in MTM zone 8 (`EPSG:2950`) so distances are in metres.

In [2]:
stations = gpd.read_file("../Data/MetroNearestGrocery.geojson").to_crs(METRIC)
stations = stations[["station", "line", "off_island", "geometry"]].reset_index(drop=True)
stores = gpd.read_file("../Data/GroceryNearestMetro.geojson").to_crs(METRIC)   # open stores only, the ones the page searches
stores = stores[["name", "address", "geometry"]].reset_index(drop=True)
stores["address"] = stores["address"].str.split().str.join(" ")   # the source pads addresses with extra spaces
print(len(stations), "stations,", len(stores), "open grocery stores")

68 stations, 1227 open grocery stores


## 2. The walking network

Only the streets near a station matter, so we download the area within 700 m of any station (the 500 m walk plus a margin) in one go, instead of the whole region. Then the network is converted to metres, so every street segment's `length` is in metres.

In [7]:
area = stations.buffer(RADIUS + 200).union_all()                     # 700 m around every station, as one shape
area = gpd.GeoSeries([area], crs=METRIC).to_crs("EPSG:4326").iloc[0]  # OSMnx downloads in degrees
G = ox.graph_from_polygon(area, network_type="walk")
G = ox.project_graph(G, to_crs=METRIC)
print(G.number_of_nodes(), "street corners,", G.number_of_edges(), "street segments")

51898 street corners, 146690 street segments


## 3. Put the stations and stores on the network

A walk starts and ends at a street corner (a **node**). Each station and store is joined to its closest node; that short straight step is added to the walk, so a store 20 m off the street still counts its 20 m.

In [8]:
# Each station and store starts its walk at the closest street corner (node).
stations["node"], stations["snap_m"] = ox.distance.nearest_nodes(G, stations.geometry.x, stations.geometry.y, return_dist=True)
stores["node"], stores["snap_m"] = ox.distance.nearest_nodes(G, stores.geometry.x, stores.geometry.y, return_dist=True)
print("Median snap to the street: stations", round(stations["snap_m"].median()), "m, stores", round(stores["snap_m"].median()), "m")

Median snap to the street: stations 10 m, stores 167 m


## 4. The walk area around each station

`ego_graph` with `distance="length"` and `radius=500` keeps every street you can reach within 500 m of walking. Each reached street is widened by 30 m on each side and the pieces are merged into one area; holes inside it are filled, because a block you can walk all the way around is within reach too. Unlike the circle, the area follows the streets: it stops at highways, rail yards and the mountain.

In [9]:
edges = ox.graph_to_gdfs(G, nodes=False)   # every street segment, with its shape


def walk_area(node):
    """The streets within RADIUS metres of walking from node, as one area."""
    reach = nx.ego_graph(G, node, radius=RADIUS, distance="length", undirected=True)
    keys = list(reach.edges(keys=True))
    if not keys:   # a station whose corner leads nowhere: fall back to the corner itself
        return Point(G.nodes[node]["x"], G.nodes[node]["y"]).buffer(STREET_BUFFER)
    streets = edges.loc[edges.index.intersection(keys), "geometry"]
    shape = streets.buffer(STREET_BUFFER).union_all().simplify(5)
    # Fill the holes: a block you can walk all the way around is within reach too.
    parts = getattr(shape, "geoms", [shape])
    return gpd.GeoSeries([Polygon(p.exterior) for p in parts]).union_all()


areas = stations[["station", "line", "off_island"]].copy()
areas = gpd.GeoDataFrame(areas, geometry=[walk_area(n) for n in stations["node"]], crs=METRIC)

## 5. Stores in each walk area, their walking distance and route

`gpd.sjoin` finds the stores inside each walk area. For each one, the shortest path on the network gives the real walking distance (station to its corner, along the streets, corner to store) and the route itself, following each street's shape. A store is kept only if the whole walk is 500 m or less.

**Check:** a walk can never be shorter than the straight line, so `walk < straight` must print 0.

In [10]:
j = gpd.sjoin(stores, areas[["station", "geometry"]], predicate="within")
print(len(j), "store and walk-area pairs from the spatial join")

U = G.to_undirected()   # walking works in both directions on every street


def xy(node):
    return (G.nodes[node]["x"], G.nodes[node]["y"])


routes = []
for st in stations.itertuples():
    lengths, paths = nx.single_source_dijkstra(U, st.node, cutoff=RADIUS, weight="length")
    for s in j[j["station"] == st.station].itertuples():
        if s.node not in lengths:
            continue   # inside the area but its closest corner is out of reach (e.g. across a highway)
        walk_m = st.snap_m + lengths[s.node] + s.snap_m
        if walk_m > RADIUS:
            continue
        path = paths[s.node]
        line = [st.geometry.coords[0], xy(path[0])]   # station to its corner
        for a, b in zip(path[:-1], path[1:]):          # then each street, following its real shape
            data = min(U.get_edge_data(a, b).values(), key=lambda d: d["length"])
            seg = list(data["geometry"].coords) if "geometry" in data else [xy(a), xy(b)]
            if Point(seg[0]).distance(Point(xy(a))) > 1:
                seg = seg[::-1]   # the stored shape runs the other way
            line.extend(seg)
        line.append(s.geometry.coords[0])              # corner to store
        routes.append({
            "station": st.station, "store": s.name, "address": s.address,
            "walk_m": round(walk_m), "straight_m": round(st.geometry.distance(s.geometry)),
            "geometry": LineString(line),
        })

routes = gpd.GeoDataFrame(routes, crs=METRIC)
counts = routes.groupby("station").size().rename("walk_count")
areas = areas.merge(counts, on="station", how="left").fillna({"walk_count": 0})
areas["walk_count"] = areas["walk_count"].astype(int)
print(routes["walk_m"].describe().round(0))
print(len(routes), "station and store pairs within a", RADIUS, "m walk")
print("walk < straight:", int((routes["walk_m"] < routes["straight_m"] - 1).sum()), "(must be 0)")
print("drawn route much shorter than its walk:", int((routes.length < 0.97 * routes["walk_m"]).sum()), "(must be 0)")

425 store and walk-area pairs from the spatial join
count    380.0
mean     296.0
std      135.0
min       15.0
25%      190.0
50%      316.0
75%      415.0
max      497.0
Name: walk_m, dtype: float64
380 station and store pairs within a 500 m walk
walk < straight: 0 (must be 0)
drawn route much shorter than its walk: 0 (must be 0)


## 6. Write the files for the page

In [12]:
areas.to_crs("EPSG:4326").to_file("../Data/MetroWalk500m.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)
routes.to_crs("EPSG:4326").to_file("../Data/MetroWalkRoutes.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)
for f in ["../Data/MetroWalk500m.geojson", "../Data/MetroWalkRoutes.geojson"]:
    print(f, round(os.path.getsize(f) / 1_000_000, 2), "MB")

../Data/MetroWalk500m.geojson 0.17 MB
../Data/MetroWalkRoutes.geojson 0.54 MB
